<a href="https://colab.research.google.com/github/gautamthampy/CMPE256-Group10/blob/baseline-covisitation/RecSysProjject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Setup & imports

import math
import random
from collections import Counter, defaultdict
import numpy as np

In [2]:
# 2. Load data

data = {}

file_path = "/content/train-2.txt"

with open(file_path, "r") as f:
    for line in f:
        parts = line.strip().split()
        if not parts:
            continue
        user = parts[0]
        items = parts[1:]
        data[user] = items

print("Number of users:", len(data))

Number of users: 52643


In [3]:
from collections import Counter

item_counts = Counter()
for items in data.values():
    item_counts.update(items)

print("Total unique items:", len(item_counts))
print("Example user, items:", next(iter(data.items())))

Total unique items: 91599
Example user, items: ('0', ['28261', '388', '5731', '401', '28284', '3570', '26806', '6802', '12212', '407', '22036', '29781', '16', '1789', '385', '29376', '661'])


In [4]:
# 3. Build train/validation split

random.seed(42)

train_data = {}  # user -> list of items (without held out)
val_items   = {}  # user -> single held-out item (str)

for user, items in data.items():
    if len(items) == 0:
        continue
    held_out = random.choice(items)
    remaining = [it for it in items if it != held_out]
    # if user had duplicates of held_out, they all get removed; fine for implicit data

    train_data[user] = remaining
    val_items[user] = held_out

print("Users in train_data:", len(train_data))
print("Users in val_items:", len(val_items))

Users in train_data: 52643
Users in val_items: 52643


In [6]:
# 4. NDCG@K for single held-out item per user

def ndcg_at_k_single(recommended, true_item, k=20):
    """
    recommended: list of item ids, ranked from best to worst
    true_item: the held-out item id (string)
    """
    try:
        rank = recommended.index(true_item)
    except ValueError:
        return 0.0

    if rank >= k:
        return 0.0

    # DCG with relevance 1 at position rank
    return 1.0 / math.log2(rank + 2)  # +2 because ranks are 0-based


def mean_ndcg_at_k(recommender_fn, users, val_dict, k=20):
    """
    recommender_fn(user, k) -> list of item_ids
    users: iterable of user ids
    val_dict: dict user -> true_item
    """
    scores = []
    for u in users:
        true_item = val_dict[u]
        recs = recommender_fn(u, k=k)
        scores.append(ndcg_at_k_single(recs, true_item, k))
    return sum(scores) / len(scores)

Algorithm 1: The popularity-based recommender serves as our simplest baseline model. It ranks all items globally by how frequently they appear in the training data and recommends the top-20 most popular items that a user has not already interacted with. Because it does not use any personalization or user-specific signals, every user receives nearly the same recommendations aside from items they have already seen. While extremely fast and easy to implement, this approach performs poorly on ranking metrics, as it generally fails to retrieve each user’s unique held-out item. In our leave-one-out evaluation, the popularity model achieved an NDCG@20 of 0.0047, confirming that


In [7]:
# 5. Popularity baseline

pop_counts = Counter()
for items in train_data.values():
    pop_counts.update(items)

print("Unique items in train:", len(pop_counts))
print("Top 5 most popular items:", pop_counts.most_common(5))

# Global ranked list of items by popularity
items_by_pop = [it for it, _ in pop_counts.most_common()]

Unique items in train: 91569
Top 5 most popular items: [('896', 1681), ('43756', 1352), ('7897', 1207), ('36454', 1123), ('36477', 1098)]


In [8]:
def recommend_popularity(user, k=20):
    """
    Recommend top-k most popular items the user has NOT seen in train_data.
    """
    seen = set(train_data[user])  # items user has interacted with in train set
    recs = []
    for it in items_by_pop:
        if it in seen:
            continue
        recs.append(it)
        if len(recs) == k:
            break
    return recs

In [9]:
users_list = list(train_data.keys())

pop_ndcg_20 = mean_ndcg_at_k(recommend_popularity, users_list, val_items, k=20)
print("Popularity NDCG@20:", pop_ndcg_20)

Popularity NDCG@20: 0.0047054802836267616
